<a href="https://colab.research.google.com/github/davidlealo/sic_ia_2026/blob/main/clases/sic_2026_ia_Clase_23.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# SIC 2026 · Inteligencia Artificial
## Clase 23: revisión de avances y calendario de cierre

Martes 24 de noviembre de 2026 · Sesión de revisión por equipos

Cada equipo se conectó a mostrar su avance y recibir retroalimentación. Este notebook reúne el calendario de cierre, las ideas que salieron en la revisión y una prueba rápida de detección de noticias falsas en español.

## 1. Calendario de cierre

| Fecha | Qué pasa |
|---|---|
| **Jueves 26 de noviembre** | **Última sesión de revisión de avances.** Si quieren que el profesor vea algo antes de presentar, es ese día. |
| Sábado 28 de noviembre | **No hay clase.** Las dudas, por Discord o correo. |
| **Lunes 30 de noviembre y martes 1 de diciembre** | **Presentaciones del proyecto final.** Si su proyecto está listo, presenten ahí. |
| Jueves 3 de diciembre | Fecha máxima para presentar. |
| **10 de diciembre** | Plazo para el **informe final** y lo que falte. Hasta esa fecha pueden mostrar el informe para recibir comentarios. |
| Diciembre | El examen final queda abierto todo el mes. |

Consultas sobre inasistencias: por correo al profesor.

## 2. Ideas que salieron en la revisión

**El modelo clasifica, pero no dice qué hacer.** Un equipo que clasifica residuos para reciclaje se dio cuenta de que la clase predicha no basta para recomendar qué hacer con el objeto. Su solución: combinar el clasificador con un **RAG** sobre documentación verificable de reciclaje en Chile, y pasarle a un modelo de lenguaje la clasificación, los documentos recuperados y la observación del usuario.

**Los errores muestran qué datos faltan.** El mismo equipo detectó que el modelo confunde una taza con diseño con textil y una pila botón con metal (43%). La causa no es el modelo sino el dataset: una clase sobrerrepresentada, imágenes repetidas y tipos de objetos sin ejemplos. Revisen la distribución de sus clases antes de seguir entrenando (celda de abajo).

**Explicar qué vio el modelo.** Otro equipo agregó una segunda herramienta para la explicabilidad: además de CLIP (similitud general entre dos imágenes), un modelo que identifica **qué objetos concretos** encontró en común, para que una persona pueda revisar el resultado.

**Detalle final.** Cuando el prototipo ya funciona, lo que queda es ingeniería de detalle: que se vea bien y que el código que van a mostrar esté ordenado para entenderlo rápido.

In [ ]:
import pandas as pd

# Reemplacen por las etiquetas de su dataset (una por imagen o por ejemplo)
etiquetas = ["plastico"] * 120 + ["metal"] * 90 + ["textil"] * 400 + ["organico"] * 150 + ["pila"] * 8

conteo = pd.Series(etiquetas).value_counts()
resumen = pd.DataFrame({"ejemplos": conteo, "porcentaje": (conteo / conteo.sum() * 100).round(1)})
print(resumen)

mayor, menor = conteo.max(), conteo.min()
if mayor / menor > 5:
    print(f"\nOjo: la clase más grande tiene {mayor / menor:.0f} veces más ejemplos que la más chica.")
    print("Consigan más ejemplos de las clases chicas, reduzcan la grande o usen pesos por clase (class_weight).")

## 3. Prueba rápida: detectar noticias falsas en español

Para un proyecto de PLN que no lograba detectar noticias falsas con sus primeros modelos, en clase buscamos modelos ya entrenados para esa tarea en Hugging Face. Uno de ellos es un **RoBERTa entrenado con noticias en español** (F1 cercano a 0,77): no es infalible, pero sirve como punto de partida.

Los pasos sugeridos:

1. Probar el modelo preentrenado con ejemplos propios, verdaderos y falsos.
2. **Comparar sus métricas con las del modelo que ya tenían** (por ejemplo, TF-IDF más un clasificador).
3. Si el preentrenado es mejor, hacerle **fine-tuning** con sus datos (Clase 18).
4. Ponerlo en una app (Gradio, Streamlit u otra).

Si el modelo de la celda no está disponible, busquen en Hugging Face `fake news spanish` y cambien `MODELO`.

In [ ]:
!pip -q install transformers

In [ ]:
from transformers import pipeline

MODELO = "Narrativaai/fake-news-detection-spanish"  # RoBERTa-large-bne revisado en clase (etiquetas REAL / FAKE)
detector = pipeline("text-classification", model=MODELO, truncation=True)

# El modelo se entrenó con "titular [SEP] texto de la noticia"
ejemplos = [
    ("El agua potable de Santiago contiene microchips",
     "Un estudio revela que el agua potable de Santiago contiene microchips de rastreo instalados por el gobierno.", "FAKE"),
    ("Banco Central mantiene la tasa de interés",
     "El Banco Central mantuvo la tasa de política monetaria en su última reunión, en línea con lo esperado por el mercado.", "REAL"),
    ("Comer limón cura todas las enfermedades",
     "Científicos confirman que comer limón en ayunas cura todas las enfermedades en una semana.", "FAKE"),
]
for titular, texto, esperado in ejemplos:
    r = detector(f"{titular} [SEP] {texto}")[0]
    ok = "✓" if r["label"] == esperado else "✗"
    print(f"{ok} {r['label']} ({r['score']:.0%}) · esperado: {esperado} · {titular}")

No es infalible: en clase se equivocó con algunas noticias verdaderas. Por eso midan con **su** conjunto de prueba, no solo con tres ejemplos.

## Dudas

Por Discord o por correo.